In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64
import os
from pathlib import Path
import pandas as pd

from CRUD_Python_Module import AnimalShelter

JupyterDash.infer_jupyter_proxy_config()

###########################
# Data Manipulation / Model
###########################
username = "aacuser"
password = "SNHU123456"

# Connect to database via CRUD module. Credentials are supplied to the class
# instead of being hardcoded inside the CRUD layer.
db = AnimalShelter(username, password)


def records_to_frame(records):
    """Convert MongoDB records to a DataFrame that Dash can display."""
    frame = pd.DataFrame.from_records(records)
    if "_id" in frame.columns:
        frame.drop(columns=["_id"], inplace=True)
    return frame


# Sending an empty query returns all animal records.
df = records_to_frame(db.read({}))


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)


def load_logo():
    logo_path = Path("Grazioso Salvare Logo.png")
    if not logo_path.exists():
        return None

    with logo_path.open("rb") as logo_file:
        return base64.b64encode(logo_file.read()).decode()


logo_image = load_logo()
branding = []
if logo_image:
    branding.append(
        html.Img(
            src="data:image/png;base64,{}".format(logo_image),
            style={"height": "100px"},
        )
    )
branding.append(html.Div("John Rosario", style={"fontSize": "16px", "marginTop": "5px"}))

app.layout = html.Div([
    html.Div(branding, style={"textAlign": "center"}),
    html.Center(html.B(html.H1("CS-340 Dashboard"))),
    html.Hr(),
    html.Div([
        html.Label("Rescue Type Filter", style={"fontWeight": "bold"}),
        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Reset (All)", "value": "reset"},
                {"label": "Water Rescue", "value": "water"},
                {"label": "Mountain / Wilderness Rescue", "value": "mountain"},
                {"label": "Disaster / Individual Tracking", "value": "disaster"},
            ],
            value="reset",
            inline=True,
        ),
    ], style={"margin": "10px 0"}),
    html.Hr(),
    dash_table.DataTable(
        id="datatable-id",
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict("records"),
        row_selectable="single",
        selected_rows=[],
        selected_columns=[],
        page_size=10,
        sort_action="native",
        filter_action="native",
        style_table={"overflowX": "auto"},
    ),
    html.Br(),
    html.Hr(),
    html.Div(
        className="row",
        style={"display": "flex"},
        children=[
            html.Div(id="graph-id", className="col s12 m6"),
            html.Div(id="map-id", className="col s12 m6"),
        ],
    ),
])


#############################################
# Interaction Between Components / Controller
#############################################
WATER_BREEDS = [
    "Labrador Retriever Mix",
    "Labrador Retriever",
    "Chesapeake Bay Retriever",
    "Newfoundland",
]

MOUNTAIN_BREEDS = [
    "German Shepherd",
    "Alaskan Malamute",
    "Old English Sheepdog",
    "Siberian Husky",
    "Rottweiler",
]

DISASTER_BREEDS = [
    "Doberman Pinscher",
    "German Shepherd",
    "Golden Retriever",
    "Bloodhound",
    "Rottweiler",
]


def build_query(filter_type):
    if filter_type == "water":
        return {
            "animal_type": "Dog",
            "breed": {"$in": WATER_BREEDS},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
        }

    if filter_type == "mountain":
        return {
            "animal_type": "Dog",
            "breed": {"$in": MOUNTAIN_BREEDS},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
        }

    if filter_type == "disaster":
        return {
            "animal_type": "Dog",
            "breed": {"$in": DISASTER_BREEDS},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300},
        }

    return {}


@app.callback(Output("datatable-id", "data"), [Input("filter-type", "value")])
def update_dashboard(filter_type):
    query = build_query(filter_type)
    filtered_frame = records_to_frame(db.read(query))
    return filtered_frame.to_dict("records")


@app.callback(Output("graph-id", "children"), [Input("datatable-id", "derived_virtual_data")])
def update_graphs(view_data):
    if not view_data:
        return []

    current_frame = pd.DataFrame.from_dict(view_data)
    if current_frame.empty or "breed" not in current_frame.columns:
        return []

    breed_counts = (
        current_frame["breed"]
        .fillna("Unknown")
        .value_counts()
        .nlargest(10)
        .reset_index()
    )
    breed_counts.columns = ["breed", "count"]

    fig = px.pie(
        breed_counts,
        names="breed",
        values="count",
        title="Top 10 Breeds (Current Table View)",
    )
    return [dcc.Graph(figure=fig)]


@app.callback(Output("datatable-id", "style_data_conditional"), [Input("datatable-id", "selected_columns")])
def update_styles(selected_columns):
    return [{
        "if": {"column_id": column},
        "background_color": "#D2F3FF",
    } for column in selected_columns]


@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows"),
    ],
)
def update_map(view_data, selected_rows):
    if not view_data:
        return []

    current_frame = pd.DataFrame.from_dict(view_data)
    if current_frame.empty:
        return []

    row = selected_rows[0] if selected_rows else 0
    if row >= len(current_frame):
        row = 0

    breed = str(current_frame.loc[row, "breed"]) if "breed" in current_frame.columns else "Unknown"
    name = str(current_frame.loc[row, "name"]) if "name" in current_frame.columns else "Unknown"

    default_center = [30.75, -97.48]
    try:
        lat = float(current_frame.loc[row, "location_lat"])
        lon = float(current_frame.loc[row, "location_long"])
        center = [lat, lon]
    except Exception:
        center = default_center
        lat, lon = default_center

    return [
        dl.Map(
            style={"width": "100%", "height": "500px"},
            center=center,
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(breed),
                        dl.Popup([html.H1("Animal Name"), html.P(name)]),
                    ],
                ),
            ],
        )
    ]


In [ ]:
app.run_server(debug=False)
